In [2]:
# HTML to Strapi Schema Plan

#This notebook tests the AI agent MVP flow:

#HTML page → section detection → CMS plan generation → validation report

In [3]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.section_detector import analyze_html_file
from src.schema_planner import llm_section_planner_node
from src.schema_validator import validate_cms_plan

In [4]:
sample_html_path = Path("sample-html/landing-page-1.html")
sample_html_path.exists()

True

In [5]:
analysis = analyze_html_file(sample_html_path)
print(json.dumps(analysis["page"], indent=2))

{
  "title": "LaunchFlow - Modern Landing Page Platform",
  "rootTag": "main",
  "sectionCount": 6
}


In [6]:
section_summary = [
    {
        "index": section["index"],
        "semanticHint": section["semanticHint"],
        "heading": section["heading"],
        "signals": section["structureSignals"],
    }
    for section in analysis["candidateSections"]
]

print(json.dumps(section_summary, indent=2))

[
  {
    "index": 1,
    "semanticHint": "hero",
    "heading": "Build beautiful landing pages without rebuilding your CMS",
    "signals": {
      "hasButtons": true,
      "hasCards": false,
      "hasTable": false,
      "hasFaqPattern": false,
      "hasForm": false,
      "hasImages": true,
      "childBlockCount": 2
    }
  },
  {
    "index": 2,
    "semanticHint": "feature",
    "heading": "Everything your team needs to launch",
    "signals": {
      "hasButtons": false,
      "hasCards": true,
      "hasTable": false,
      "hasFaqPattern": false,
      "hasForm": false,
      "hasImages": false,
      "childBlockCount": 3
    }
  },
  {
    "index": 3,
    "semanticHint": "testimonial",
    "heading": "Trusted by growing teams",
    "signals": {
      "hasButtons": false,
      "hasCards": true,
      "hasTable": false,
      "hasFaqPattern": false,
      "hasForm": false,
      "hasImages": false,
      "childBlockCount": 2
    }
  },
  {
    "index": 4,
    "semanticHint"

In [12]:
state = llm_section_planner_node({
    "html_analysis": analysis,
    "errors": [],
})

if state.get("errors"):
    raise RuntimeError(state["errors"])

cms_plan = state["cms_plan"]
print(json.dumps(cms_plan["pageModel"], indent=2))

{
  "kind": "singleType",
  "apiName": "launch-flow-landing-page",
  "displayName": "LaunchFlow Landing Page",
  "singularName": "launch-flow-landing-page",
  "pluralName": "launch-flow-landing-pages",
  "description": "Single type for the LaunchFlow marketing landing page."
}


In [13]:
validation_report = validate_cms_plan(cms_plan)
print(json.dumps(validation_report, indent=2))

{
  "isValid": true,
  "errors": [],
  "warnings": [
    "landing-page.pricing-card.features: repeatable scalar fields may need a repeatable component in Strapi",
    "globalBlocks.header: componentPlan is not set",
    "globalBlocks.footer: componentPlan is not set",
    "Contact form fields are stored as structured JSON in the contact component; Strapi does not natively model dynamic form field schemas, so consider a dedicated form builder or custom component if field-level editing is required.",
    "Added missing seo singleTypeAttribute for seedData.seo."
  ],
  "summary": {
    "pageApiName": "launch-flow-landing-page",
    "componentCount": 10,
    "singleTypeAttributeCount": 7,
    "seedDataKeyCount": 7,
    "componentUids": [
      "landing-page.hero",
      "landing-page.feature-card",
      "landing-page.features",
      "landing-page.testimonial-card",
      "landing-page.testimonials",
      "landing-page.pricing-card",
      "landing-page.pricing",
      "landing-page.faq-

In [14]:
components_summary = [
    {
        "uid": component["uid"],
        "fields": [field["name"] for field in component["fields"]],
    }
    for component in cms_plan["components"]
]

print(json.dumps(components_summary, indent=2))

[
  {
    "uid": "landing-page.hero",
    "fields": [
      "eyebrow",
      "title",
      "description",
      "primary_cta_label",
      "primary_cta_href",
      "secondary_cta_label",
      "secondary_cta_href",
      "image"
    ]
  },
  {
    "uid": "landing-page.feature-card",
    "fields": [
      "title",
      "description"
    ]
  },
  {
    "uid": "landing-page.features",
    "fields": [
      "title",
      "description",
      "feature_cards"
    ]
  },
  {
    "uid": "landing-page.testimonial-card",
    "fields": [
      "quote",
      "author_name",
      "author_role"
    ]
  },
  {
    "uid": "landing-page.testimonials",
    "fields": [
      "title",
      "testimonial_cards"
    ]
  },
  {
    "uid": "landing-page.pricing-card",
    "fields": [
      "title",
      "price",
      "description",
      "features",
      "is_highlighted"
    ]
  },
  {
    "uid": "landing-page.pricing",
    "fields": [
      "title",
      "pricing_cards"
    ]
  },
  {
    "uid": "la

In [15]:
print(json.dumps(cms_plan, indent=2))

{
  "pageModel": {
    "kind": "singleType",
    "apiName": "launch-flow-landing-page",
    "displayName": "LaunchFlow Landing Page",
    "singularName": "launch-flow-landing-page",
    "pluralName": "launch-flow-landing-pages",
    "description": "Single type for the LaunchFlow marketing landing page."
  },
  "seo": {
    "enabled": true,
    "component": "shared.seo"
  },
  "globalBlocks": {
    "header": {
      "handling": "global_single_type",
      "apiName": "global-header",
      "componentPlan": null
    },
    "footer": {
      "handling": "global_single_type",
      "apiName": "global-footer",
      "componentPlan": null
    }
  },
  "components": [
    {
      "uid": "landing-page.hero",
      "category": "landing-page",
      "displayName": "Hero",
      "fileName": "hero",
      "fields": [
        {
          "name": "eyebrow",
          "type": "string",
          "required": false,
          "component": null,
          "repeatable": false,
          "multiple": false,